# Volume 12, Chapter 5: Cyclotomic fields and Fermat (Python companion)

Exercises first, solutions at the end. We check the discriminant formula of Theorem 5.3, watch primes split in $\mathbb Q(\zeta_n)$, meet the Gauss sum numerically, hunt irregular primes with Bernoulli numbers, catch Kummer's non-principal ideal in $\mathbb Z[\zeta_{23}]$, count Sophie Germain primes, and finish with the computer-science star of the chapter: the **number-theoretic transform**, the fast Fourier transform over $\mathbb F_q$, which exists exactly when $q$ splits completely in a cyclotomic field. Requires `cypari2` and `numpy`.

In [ ]:
import math, cmath, random, time
import numpy as np
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)

## Exercises

**P5.1 (★) Discriminants.** For every prime power $p^k\le 200$ with $p^k\ne2$, compare PARI's `nfdisc(polcyclo(p^k))` with $\pm p^{p^{k-1}(k(p-1)-1)}$, sign $(-1)^{\varphi(p^k)/2}$ (Theorem 5.3).

**P5.2 (★) Splitting in $\mathbb Q(\zeta_n)$.** For $n=5,7,12$ and the primes $p<60$, list the residue degrees of the primes above $p$ (PARI's `idealprimedec`) and compare with Corollary 5.5: $f$ is the order of $p$ modulo $n$, and $e=\varphi(p^k)$ for $p^k\,\|\,n$.

**P5.3 (★) The Gauss sum.** Compute $g=\sum_a\left(\frac ap\right)\zeta_p^a$ numerically for the odd primes $p<50$. Check $g^2=p^*$, and look at $g$ itself: what do you notice about its sign? Then check Theorem 5.7 for all pairs of odd primes below $200$.

**P5.4 (★★) Irregular primes.** Using Kummer's criterion, list the irregular primes below $5000$ and the indices $k$ with $p\mid B_k$. What fraction of primes is regular? Compare with Siegel's prediction $e^{-1/2}$. Which prime below $5000$ divides the most Bernoulli numbers?

**P5.5 (★★) Class numbers of $\mathbb Q(\zeta_p)$.** Compute $h_p$ for $p\le23$, and check Exercise 5.7: the primes of $\mathbb Z[\zeta_{23}]$ above $47$ are not principal, but their cubes are.

**P5.6 (★★) Sophie Germain primes.** Count the primes $p<x$ such that $2p+1$ is prime, for $x=10^3,\dots,10^7$, and compare with the Hardy–Littlewood prediction $2C_2\int_2^x\frac{dt}{\log t\,\log(2t+1)}$, $C_2=\prod_{p>2}\bigl(1-\frac1{(p-1)^2}\bigr)\approx0.66016$. (Nobody has proved that there are infinitely many.)

**P5.7 (★★★) CS application: the number-theoretic transform.** Let $q$ be a prime with $q\equiv1\pmod N$, $N=2^m$.
1. Explain why $\mathbb F_q$ then contains a primitive $N$-th root of unity $w$, and why $q\equiv1\pmod N$ means that $q$ splits completely in $\mathbb Q(\zeta_N)$.
2. Implement the iterative radix-2 NTT over $\mathbb F_q$ and its inverse, for $q=998\,244\,353=119\cdot2^{23}+1$, and use it to multiply polynomials. Check against the schoolbook product, and time both for degrees $2^{8},\dots,2^{14}$.
3. The post-quantum standard ML-KEM (Kyber) works in $\mathbb F_{3329}[X]/(X^{256}+1)$. Show that $3329$ splits completely in $\mathbb Q(\zeta_{256})$ but not in $\mathbb Q(\zeta_{512})$, and deduce how $X^{256}+1$ factors modulo $3329$. Why does Kyber use an "incomplete" NTT?

## Solutions

### P5.1
**How to find it.** `polcyclo(n)` is $\Phi_n$, and `nfdisc` computes $d_K$ independently of our theorem (by Round 2/Round 4 algorithms), so the comparison is a genuine test of $\mathcal O_K=\mathbb Z[\zeta]$ and of the formula.

In [ ]:
bad = 0
for n in range(3, 201):
    fa = pari.factor(n)
    if len(fa[0]) != 1:
        continue
    p, k = int(fa[0][0]), int(fa[1][0])
    phi = (p - 1) * p**(k - 1)
    pred = (-1)**(phi // 2) * p**(p**(k - 1) * (k * (p - 1) - 1))
    d = int(pari.nfdisc(pari.polcyclo(n)))
    bad += (d != pred)
    if n in (3, 4, 5, 7, 8, 9, 16, 25, 27):
        print(f"{n:4d}: d = {d}")
print("mismatches:", bad)

No mismatch. Note $d_{\mathbb Q(\zeta_8)}=256$ and $d_{\mathbb Q(\zeta_9)}=-3^9$, as in Exercise 5.4.

### P5.2
**How to find it.** `idealprimedec(nf, p)` returns the primes above $p$ with their $e$ and $f$; the prediction is the multiplicative order of $p$ modulo the prime-to-$p$ part of $n$.

In [ ]:
def predicted(n, p):
    k = 0
    m = n
    while m % p == 0:
        m //= p; k += 1
    e = (p - 1) * p**(k - 1) if k else 1
    f = int(pari.znorder(pari.Mod(p, m))) if m > 1 else 1
    phi = int(pari.eulerphi(n))
    return e, f, phi // (e * f)
for n in (5, 7, 12):
    nf = pari.nfinit(pari.polcyclo(n))
    rows, ok = [], True
    for p in pari.primes([2, 60]):
        p = int(p)
        dec = pari.idealprimedec(nf, p)
        e, f, g = int(dec[0][2]), int(dec[0][3]), len(dec)
        ok &= ((e, f, g) == predicted(n, p))
        rows.append(f"{p}:(e{e},f{f},g{g})")
    print(f"n={n}:", ok)
    print("   ", " ".join(rows[:10]))

For $n=12$, $f$ is the order of $p$ in $(\mathbb Z/12\mathbb Z)^*\cong(\mathbb Z/2)^2$, so it is $1$ or $2$, and every $p\ge5$ splits into $2$ or $4$ primes: $p\equiv1\pmod{12}$ splits completely (e.g. $13, 37$), and the other residues give $f=2$. The primes $2$ and $3$ ramify with $e=2$.

### P5.3
**How to find it.** Sum with complex exponentials; the Legendre symbol is `kronecker`. For the second part compare $\bigl(\frac{p^*}q\bigr)$ and $\bigl(\frac qp\bigr)$ directly.

In [ ]:
for p in pari.primes([3, 50]):
    p = int(p)
    g = sum(int(pari.kronecker(a, p)) * cmath.exp(2j * math.pi * a / p) for a in range(1, p))
    pstar = p if p % 4 == 1 else -p
    print(f"p={p:2d}  g = {g.real:+.6f} {g.imag:+.6f}i   g^2 = {(g*g).real:+.3f}   p* = {pstar}")
ps = [int(p) for p in pari.primes([3, 200])]
print("reciprocity holds for all pairs:",
      all(int(pari.kronecker(p if p % 4 == 1 else -p, q)) == int(pari.kronecker(q, p)) for p in ps for q in ps if p != q))

Always $g=+\sqrt p$ when $p\equiv1\pmod4$ and $g=+i\sqrt p$ when $p\equiv3\pmod4$: the sign is never negative. Gauss conjectured this in 1801 and needed until 1805 to prove it ("not a week passed in which I did not try in vain to untie this knot", he wrote to Olbers). The square $g^2=p^*$ is algebra; the sign of $g$ is analysis, and it is the deepest part of the story. Proofs are in Ireland–Rosen, Chapter 6, and via Poisson summation in Chapter 9 of this book.

### P5.4
**How to find it.** PARI's `bernfrac(k)` is fast for $k$ up to a few thousand. Compute the numerators once and reduce them modulo each $p$, for even $k$ from $2$ to $p-3$.

In [ ]:
N = 5000
t = time.perf_counter()
num = {k: int(pari.numerator(pari.bernfrac(k))) for k in range(2, N, 2)}
primes = [int(p) for p in pari.primes([3, N])]
irr = {p: [k for k in range(2, p - 2, 2) if num[k] % p == 0] for p in primes}
irregular = [p for p in primes if irr[p]]
print("time %.1f s" % (time.perf_counter() - t))
print("irregular primes below 200:", [(p, irr[p]) for p in irregular if p < 200])
frac = 1 - len(irregular) / len(primes)
print(f"{len(irregular)} irregular out of {len(primes)}: regular fraction {frac:.4f}, e^(-1/2) = {math.exp(-0.5):.4f}")
best = max(primes, key=lambda p: len(irr[p]))
print("most Bernoulli divisors:", best, irr[best])

About $61\%$ of the primes below $5000$ are regular, close to $e^{-1/2}\approx0.6065$. Siegel's heuristic: $p$ divides each of the $\frac{p-3}2$ numerators with "probability" $\frac1p$, independently, so the chance of dividing none is $(1-\frac1p)^{(p-3)/2}\approx e^{-1/2}$. The number of such indices $k$ is the *index of irregularity* of $p$: Kummer's own example $157$ has index $2$, and below $5000$ the record is $491$, with index $3$ ($B_{292}$, $B_{336}$, $B_{338}$).

### P5.5
**How to find it.** `bnfinit(polcyclo(p), 1)` computes the class group (assuming GRH for speed; for these small fields `bnfcertify` can remove the assumption). `bnfisprincipal(bnf, P, 0)` returns the class of $P$ in the class group as a vector of exponents; the zero vector means principal.

In [ ]:
for p in (3, 5, 7, 11, 13, 17, 19, 23):
    bnf = pari.bnfinit(pari.polcyclo(p), 1)
    print(p, "h =", bnf.bnf_get_no(), " class group", bnf.bnf_get_cyc())
bnf = pari.bnfinit(pari.polcyclo(23), 1)
P = pari.idealprimedec(bnf, 47)[0]
print("norm of P:", pari.idealnorm(bnf, P))
print("class of P:", pari.bnfisprincipal(bnf, P, 0), " class of P^3:", pari.bnfisprincipal(bnf, pari.idealpow(bnf, P, 3), 0))
print("bnfcertify:", pari.bnfcertify(bnf))

$h_p=1$ for $p\le19$ and $h_{23}=3$; the class of $P$ is a generator of the cyclic group of order $3$, and $P^3$ is principal. `bnfcertify` returns 1, so the result does not depend on GRH. (Montgomery and Uchida proved in 1971–1976 that $p\le19$ are the only primes with $h_p=1$.)

### P5.6
**How to find it.** Sieve the primes up to $2x+1$ once, then count. The integral is computed numerically.

In [ ]:
X = 10**7
sieve = np.ones(2 * X + 2, dtype=bool); sieve[:2] = False
for i in range(2, int((2 * X + 1)**0.5) + 1):
    if sieve[i]:
        sieve[i*i::i] = False
ps = np.nonzero(sieve[:X])[0]
C2 = 1.0
for p in np.nonzero(sieve[:10**6])[0][1:]:
    C2 *= 1 - 1 / (p - 1)**2
print("C2 =", C2)
def li2(x, steps=200000):
    t = np.linspace(2, x, steps)
    y = 1 / (np.log(t) * np.log(2 * t + 1))
    return np.trapezoid(y, t)
for e in range(3, 8):
    x = 10**e
    sg = ps[ps < x]
    count = int(np.count_nonzero(sieve[2 * sg + 1]))
    pred = 2 * C2 * li2(x)
    print(f"x=10^{e}: {count:7d} Sophie Germain primes, prediction {pred:9.0f}, ratio {count/pred:.4f}")

The prediction is excellent (ratio within half a percent at $10^7$). By Exercise 5.6 each Sophie Germain prime gives a free proof of the first case of Fermat for its exponent; but proving that there are infinitely many of them is as hard as the twin prime conjecture.

### P5.7
**How to find it.** (1) $\mathbb F_q^*$ is cyclic of order $q-1$, so it has an element of order $N$ iff $N\mid q-1$; and by Theorem 2.19 the residue degree of $q$ in $\mathbb Q(\zeta_N)$ is the order of $q$ modulo $N$, which is $1$ iff $q\equiv1\pmod N$. Both say: $\mathcal O_K/Q\cong\mathbb F_q$ for every prime $Q$ above $q$, and the image of $\zeta_N$ is a primitive $N$-th root of unity in $\mathbb F_q$. (2) The Cooley–Tukey FFT only uses $w^{N/2}=-1$ and $w^N=1$, so it works verbatim in $\mathbb F_q$, with exact arithmetic and no rounding.

In [ ]:
Q = 998244353            # 119 * 2^23 + 1
G = 3                    # a primitive root modulo Q
def ntt(a, invert=False):
    a = list(a); n = len(a)
    j = 0
    for i in range(1, n):          # bit-reversal permutation
        bit = n >> 1
        while j & bit:
            j ^= bit; bit >>= 1
        j |= bit
        if i < j:
            a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        w = pow(G, (Q - 1) // length, Q)
        if invert:
            w = pow(w, Q - 2, Q)
        half = length // 2
        ws = [1] * half
        for k in range(1, half):
            ws[k] = ws[k - 1] * w % Q
        for s in range(0, n, length):
            for k in range(half):
                u, v = a[s + k], a[s + k + half] * ws[k] % Q
                a[s + k], a[s + k + half] = (u + v) % Q, (u - v) % Q
        length <<= 1
    if invert:
        ninv = pow(n, Q - 2, Q)
        a = [x * ninv % Q for x in a]
    return a
def mul_ntt(f, g):
    n = 1
    while n < len(f) + len(g) - 1:
        n <<= 1
    F, Gv = ntt(f + [0] * (n - len(f))), ntt(g + [0] * (n - len(g)))
    return ntt([x * y % Q for x, y in zip(F, Gv)], invert=True)[:len(f) + len(g) - 1]
def mul_school(f, g):
    h = [0] * (len(f) + len(g) - 1)
    for i, x in enumerate(f):
        for j, y in enumerate(g):
            h[i + j] = (h[i + j] + x * y) % Q
    return h
random.seed(1)
for m in range(8, 15):
    n = 2**m
    f = [random.randrange(Q) for _ in range(n)]; g = [random.randrange(Q) for _ in range(n)]
    t = time.perf_counter(); h1 = mul_ntt(f, g); t1 = time.perf_counter() - t
    if m <= 11:
        t = time.perf_counter(); h2 = mul_school(f, g); t2 = time.perf_counter() - t
        print(f"deg 2^{m}: NTT {t1:.3f} s, schoolbook {t2:.3f} s, equal: {h1 == h2}")
    else:
        print(f"deg 2^{m}: NTT {t1:.3f} s, schoolbook (estimated) {t2 * 4**(m - 11):.1f} s")

The NTT costs $O(N\log N)$ multiplications against $N^2$ for the schoolbook method; even in pure Python the crossover comes early, and at degree $16384$ the NTT wins by two orders of magnitude.

(3) Kyber's prime:

In [ ]:
q = 3329
print("q - 1 =", pari.factor(q - 1))
for N in (256, 512):
    print(f"q mod {N} = {q % N}; residue degree of q in Q(zeta_{N}) = order of q mod {N} =", pari.znorder(pari.Mod(q, N)))
fa = pari.factormod(pari("x^256+1"), q)
print("X^256+1 mod 3329 factors into", len(fa[0]), "factors of degree", set(int(pari.poldegree(f)) for f in fa[0]))

$3328=2^8\cdot13$, so $3329\equiv1\pmod{256}$ but $3329\equiv257\pmod{512}$: $3329$ splits completely in $\mathbb Q(\zeta_{256})$, into $\varphi(256)=128$ primes of degree $1$, but in $\mathbb Q(\zeta_{512})$ it splits into $128$ primes of degree $2$. Now $X^{256}+1=\Phi_{512}$, so by Kummer–Dedekind it factors modulo $3329$ exactly like $3329$ in $\mathbb Z[\zeta_{512}]$: into $128$ irreducible quadratics $X^2-\zeta^{2i+1}$, with $\zeta$ a primitive $256$-th root of unity in $\mathbb F_{3329}$. So the "NTT" of Kyber stops one level early and multiplies the $128$ residues as linear polynomials modulo quadratics. Among the primes $\equiv1\pmod{256}$, $3329$ is the third, after $257$ and $769$, and the smallest large enough for Kyber's security and decryption-error targets. The splitting law of cyclotomic fields is, quite literally, a design parameter of the cryptography that protects internet traffic since 2024.